<a href="https://colab.research.google.com/github/serelk/nebius-academy-course/blob/main/week3/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
from google.colab import userdata


In [2]:
dataset = load_dataset("PatronusAI/financebench",)

README.md: 0.00B [00:00, ?B/s]

financebench_merged.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

In [25]:
df = dataset["train"].to_pandas()



,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,None,What is the FY2018 capital expenditure amount ...,$1577.00,The metric capital expenditures was directly e...,OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,None,Assume that you are a public equities analyst....,$8.70,"The metric ppne, net was directly extracted fr...",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
2,financebench_id_00499,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",CAPEX/Revenue\nFixed Assets/Total Assets\nROA=...,OPEN_SOURCE,[{'evidence_text': '3M Company and Subsidiarie...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
3,financebench_id_01226,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasonin...,dg17,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,None,OPEN_SOURCE,"[{'evidence_text': 'SG&A, measured as a percen...",Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
4,financebench_id_01865,3M,3M_2022_10K,novel-generated,None,None,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,None,OPEN_SOURCE,[{'evidence_text': 'Worldwide Sales Change By...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...


In [23]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=userdata.get('NEBIUS_API_KEY')
)

In [32]:
def get_response(question):

  SYSTEM_PROMPT = """You are usefull assistant . Please provide a short and consice unswer starting with Yes/No
                      For example
                      Yes. XXXXX's capital intensity ratio was approximately 2.774729 """
  response = client.chat.completions.create(
      model="meta-llama/Llama-3.3-70B-Instruct",
      messages=[
          {
            "role": "system",
            "content": f"{SYSTEM_PROMPT}"
            },
          {

              "role": "user",
              "content": [
                  {
                      "type": "text",
                      "text": f"{question}"
                  }
              ]
          }
      ]
  )

  return response.choices[0].message.content

In [36]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
filtered_df = df.where(df.question_type == "domain-relevant").dropna().sort_values(by="financebench_id").head(5)
filtered_df["answer"]
print("Iterating over filtered_df:")
for index, row in filtered_df.head(5).iterrows():
    print(f"ID: {row['financebench_id']}, Question: {row['question']}")
    response = get_response(row['question'])
    print(f"Response: {response}")
    print(f"Ground Truth: {row['answer']}")
    print("\n")

Iterating over filtered_df:
ID: financebench_id_00005, Question: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
Response: Yes. Corning's working capital was approximately $3.4 billion based on FY2022 data, indicating a positive working capital position.
Ground Truth: Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers only operating current assets and current liabilities that were clearly shown in the balance sheet.


ID: financebench_id_00070, Question: Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
Response: Yes. American Water Works had positive working capital based on FY2022 data.
Ground Truth: No, American Water Works had negative working capital of -$1561M 